# State Space Models

> ⚡Compute Note: You can run this notebook on CPU. 

Large Language Models (LLMs) keep getting longer context windows — 8k, 32k, 128k, and beyond. But underneath this progress lies a fundamental limitation: Transformers scale quadratically with sequence length. That means doubling the context quadruples the compute and memory footprint.

This makes extremely long-context modeling expensive, slow, and sometimes impossible to train on standard hardware. As we continue pushing towards models that can reason over entire books, multi-hour conversations, or days of sensor data, we need architectures that can scale.

That brings us to State Space Models (SSMs) — a class of sequence models that can process arbitrarily long inputs with:
* O(N) compute
* O(1) memory
* Effectively infinite context length

Unlike Transformers, which store all past tokens in a growing KV cache, SSMs maintain a compressed hidden state that evolves over time and summarizes the entire history. For more information, check [this](https://huggingface.co/blog/lbourdois/get-on-the-ssm-train) out. 

In this tutorial, we will build intuition about:
* Why Transformers struggle with long context
* How classical State Space Models work
* How modern variants like Mamba add selectivity and nonlinearity
* Why SSMs are becoming central to long-context LLM research (Mamba, Jamba, Hyena, RWKV, RetNet)

We will start from the classical state-space equations, discretize them for deep learning, and gradually build toward a minimal working implementation of a Selective SSM. We will compare this with MLPs, RNNs, and a tiny Transformer on a long-range synthetic task in which Transformers struggle but SSMs succeed.

Let’s begin by understanding why Transformers struggle as context lengths grow.


In [1]:
import math
import random
from dataclasses import dataclass
from typing import Optional, Tuple, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


def set_seed(seed: int = 42) -> None:
    """Fix random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)

Using device: cpu


## 🧪 A Toy Problem That Requires Understanding Long-Range Structure

To understand why State Space Models matter, we need a task that highlights how different architectures handle repeating patterns with long periods — something that requires tracking structure far back in the sequence, not just recent context.

Instead of a brutal copy-from-200-steps-ago task, we use a smoother but still challenging version of long-range dependence: **🔁 Periodic Sequence Prediction Task**.

We create sequences with a hidden periodic structure. For each training example:
1.	We randomly choose a period P between 10 and 100
2.	We generate a random base pattern: $[b_0, b_1, \ldots, b_{P-1}]$
3.	We repeat this pattern until we reach a full sequence of length: $T = 300$. The resulting sequence looks like:

$$x = (b_0, b_1, \ldots, b_{P-1}, b_0, b_1, \ldots, b_{P-1}, \ldots)$$

Our task is next-token prediction:

$$y_t = x_{(t+1) \bmod T}$$

This forces the model to infer — and internally represent — the periodicity of the sequence to make correct predictions.

We use:
	•	Sequence length: $T = 300$
	•	Period range: $P \in [10, 100]$
	•	Vocabulary: integers $\{0, 1, \ldots, 19\}$


❗ Why This Task Still Requires Long-Range Understanding

Although the task is gentler than the previous copy task, it still demands models to capture global structure, not just local patterns:

* MLP fails because it lacks any notion of order or repetition. It cannot infer periodicity.

* Tiny Transformer has positional information and attention, but with only a few layers and small embedding size, learning long periods (e.g., 80–100) is difficult. Attention tends to focus locally rather than on long-range repeats.

* GRU can, in principle, encode “where we are” in the cycle using its hidden state. Performs better than MLP/Transformer at moderate periods, but still struggles as periods grow large.

* State Space Models (SSMs), can track a smoothly evolving internal state that naturally captures long-range periodic structure. These models are explicitly built to maintain information over long sequences without quadratic compute.

In [20]:
SEQ_LEN = 300          # full sequence length
VOCAB_SIZE = 20        # integers from 0 to 19
TRAIN_SAMPLES = 5000
TEST_SAMPLES = 500

MIN_PERIOD = 10
MAX_PERIOD = 100


def make_periodic_sample(
    seq_len: int = SEQ_LEN,
    vocab_size: int = VOCAB_SIZE,
    min_period: int = MIN_PERIOD,
    max_period: int = MAX_PERIOD,
):
    """
    Generate one periodic sequence and its next-token targets.
    """
    # 1) Sample period
    period = random.randint(min_period, max_period)

    # 2) Sample base pattern
    pattern = torch.randint(0, vocab_size, (period,))

    # 3) Tile the pattern to length seq_len
    n_repeats = (seq_len + period - 1) // period
    x_full = pattern.repeat(n_repeats)[:seq_len]  # (seq_len,)

    # 4) Next-token prediction target (circular)
    # y[t] = x_full[(t + 1) % seq_len]
    y_full = torch.roll(x_full, shifts=-1)

    return x_full, y_full


def build_periodic_dataset(n_samples: int):
    X, Y = [], []
    for _ in range(n_samples):
        x, y = make_periodic_sample()
        X.append(x)
        Y.append(y)
    return torch.stack(X), torch.stack(Y)


X_train, Y_train = build_periodic_dataset(TRAIN_SAMPLES)
X_test, Y_test = build_periodic_dataset(TEST_SAMPLES)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: torch.Size([5000, 300])
Test shape: torch.Size([500, 300])


## 🔹 Baseline 1: A Simple MLP

To establish a baseline, we start with the simplest model possible:
a Multilayer Perceptron (MLP).

This MLP:
* Embeds each integer into a vector
* Flattens the entire sequence into one large vector
* Passes it through a few feedforward layers
* Predicts all outputs at once

But there’s a fundamental issue. An MLP has no idea what sequence order means. It cannot learn temporal dependencies, positional structure, or long-range relationships.

This makes the MLP a perfect “dumb baseline” to highlight why architectures with recurrence or attention are needed for sequence modeling.

Below we train the MLP and observe its performance.


In [26]:
class MLPBaseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, 32)
        self.net = nn.Sequential(
            nn.Linear(SEQ_LEN * 32, 512),
            nn.ReLU(),
            nn.Linear(512, SEQ_LEN * VOCAB_SIZE),
        )

    def forward(self, x):
        # x: (B, T)
        emb = self.embed(x)                 # (B, T, 32)
        flat = emb.reshape(x.size(0), -1)   # (B, T*32)
        out = self.net(flat)                # (B, T * VOCAB_SIZE)
        return out.reshape(x.size(0), SEQ_LEN, VOCAB_SIZE)  # (B, T, V)


mlp = MLPBaseline().to(device)
criterion = nn.CrossEntropyLoss()
mlp_optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3)

def train_mlp(model, n_steps: int = 300, batch_size: int = 64):
    model.train()
    for step in range(n_steps):
        idx = torch.randint(0, TRAIN_SAMPLES, (batch_size,))
        batch_x = X_train[idx].to(device)
        batch_y = Y_train[idx].to(device)

        logits = model(batch_x)
        loss = criterion(logits.reshape(-1, VOCAB_SIZE), batch_y.reshape(-1))

        mlp_optimizer.zero_grad()
        loss.backward()
        mlp_optimizer.step()

        if step % 50 == 0:
            print(f"[MLP] step {step} | loss = {loss.item():.4f}")

train_mlp(mlp)

[MLP] step 0 | loss = 3.0204
[MLP] step 50 | loss = 2.9145
[MLP] step 100 | loss = 2.4918
[MLP] step 150 | loss = 2.2300
[MLP] step 200 | loss = 1.7901
[MLP] step 250 | loss = 1.6016


## 🔹 Baseline 2: A Tiny Transformer (with Positional Encoding)

The next baseline is a small Transformer encoder.
Transformers are extremely powerful sequence models, but their ability to learn long-range dependencies depends heavily on:
1.	Attention capacity
2.	Depth and width of the model
3.	Proper positional information

Since attention alone does not encode order, we add sinusoidal positional encodings, exactly like in the original Attention is All You Need paper. This allows the model to distinguish whether a token occurred early or late in the sequence.

What we expect:
* The Transformer should outperform the MLP.
* The model must infer the period length and where it currently is within the repeating cycle. 
* Attention will likely become too diffuse to reliably discover the exact token that occurred 200 steps ago.

This makes the tiny Transformer a great middle baseline, it has some long-range capability, but not enough to handle variable-length periodic structure robustly.

The code below defines and trains this tiny Transformer baseline.

In [30]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=SEQ_LEN):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # shape (1, T, d_model)

    def forward(self, x):
        # x: (B, T, d_model)
        return x + self.pe[:, :x.size(1), :]


class TinyTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        d_model = 64
        n_heads = 4
        n_layers = 2

        self.embed = nn.Embedding(VOCAB_SIZE, d_model)
        self.pos_encoding = SinusoidalPositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=n_heads, 
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, 
            num_layers=n_layers
        )

        self.out = nn.Linear(d_model, VOCAB_SIZE)

    def forward(self, x):
        emb = self.embed(x)                      # (B, T, d_model)
        emb = self.pos_encoding(emb)             # add positional info
        h = self.transformer(emb)                # (B, T, d_model)
        return self.out(h)                       # (B, T, vocab)


transformer = TinyTransformer().to(device)
opt_t = torch.optim.Adam(transformer.parameters(), lr=1e-3)


def train_transformer(model, n_steps=300):
    model.train()
    for step in range(n_steps):
        idx = torch.randint(0, TRAIN_SAMPLES, (32,))
        batch_x = X_train[idx].to(device)
        batch_y = Y_train[idx].to(device)

        logits = model(batch_x)
        loss = criterion(logits.reshape(-1, VOCAB_SIZE), batch_y.reshape(-1))

        opt_t.zero_grad()
        loss.backward()
        opt_t.step()

        if step % 50 == 0:
            print(f"[Transformer] step {step} | loss = {loss.item():.4f}")


train_transformer(transformer)

[Transformer] step 0 | loss = 3.1721
[Transformer] step 50 | loss = 3.0035
[Transformer] step 100 | loss = 2.9735
[Transformer] step 150 | loss = 2.9274
[Transformer] step 200 | loss = 2.8658
[Transformer] step 250 | loss = 2.8272


### 📊 Evaluating the Baseline Models

Now that both the MLP and the tiny Transformer have been trained, we evaluate how well they perform on the Repeat-After-K long-range dependency task.

The evaluation is simple:
* Run the model on the entire test set
* Compute argmax predictions at each timestep
* Compare them with the ground truth targets
* Compute the mean accuracy over all tokens in all sequences

This gives us a clear picture of how well each architecture handles a dependency that reaches 200 steps into the past.


In [31]:
@torch.no_grad()
def evaluate(model):
    model.eval()
    x = X_test.to(device)
    y = Y_test.to(device)

    logits = model(x)
    preds = logits.argmax(dim=-1)

    mask = torch.arange(SEQ_LEN, device=device) >= REPEAT_OFFSET
    correct = (preds[:, mask] == y[:, mask]).float().mean().item()
    return correct


mlp_acc = evaluate(mlp)
transformer_acc = evaluate(transformer)

print(f"MLP Test Accuracy:          {mlp_acc:.4f}")
print(f"Transformer Test Accuracy:  {transformer_acc:.4f}")

MLP Test Accuracy:          0.2434
Transformer Test Accuracy:  0.1114


## 🔹 Baseline 3: GRU — A Recurrent Model with True Sequential Memory

Before we shift to State Space Models, it is worth examining how a recurrent neural network performs on this long-range dependency task. A GRU (Gated Recurrent Unit) processes the sequence one token at a time and maintains a hidden state that evolves with each step:

$$h_t = \mathrm{GRU}(x_t, h_{t-1})$$

This hidden state acts as a compressed memory of everything that has happened so far.
Unlike the MLP and Transformer baselines:

### ✔ Why GRUs should perform better

1. They have true temporal recurrence. The hidden state is explicitly designed to carry information across time.
In principle, a GRU can remember something seen 200 steps earlier.

2. They require only O(1) memory with respect to sequence length. There is no attention matrix, no expanding KV cache, and no need to store all past tokens.

3. Their updates are local and stable. Each step only depends on the previous state.
This often helps for tasks with simple long-range rules.

### ✘ But GRUs still struggle with very long dependencies

Even with gating, the signal must survive through hundreds of recurrent steps.
Gradients can still fade, and long-distance recall remains difficult—especially for small models like the one we use here.

This makes the GRU a stronger baseline than MLP or a tiny Transformer, but still not strong enough to solve Repeat-After-200 reliably.

Below, we implement and train a simple GRU model and evaluate its performance.


In [ ]:
class GRUBaseline(nn.Module):
    def __init__(self, d_model: int = 256, num_layers: int = 2):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, d_model)
        self.gru = nn.GRU(
            input_size=d_model,
            hidden_size=d_model,
            num_layers=num_layers,
            batch_first=True,
        )
        self.out = nn.Linear(d_model, VOCAB_SIZE)

    def forward(self, x):
        # x: (B, T)
        emb = self.embed(x)              # (B, T, d_model)
        h, _ = self.gru(emb)             # (B, T, d_model)
        logits = self.out(h)             # (B, T, vocab)
        return logits


gru = GRUBaseline().to(device)
opt_gru = torch.optim.Adam(gru.parameters(), lr=1e-3)

def train_gru(model, n_steps: int = 300, batch_size: int = 64):
    model.train()
    for step in range(n_steps):
        idx = torch.randint(0, TRAIN_SAMPLES, (batch_size,))
        batch_x = X_train[idx].to(device)
        batch_y = Y_train[idx].to(device)

        logits = model(batch_x)
        loss = criterion(logits.reshape(-1, VOCAB_SIZE), batch_y.reshape(-1))
        
        opt_gru.zero_grad()
        loss.backward()
        opt_gru.step()

        if step % 50 == 0:
            print(f"[GRU] step {step} | loss = {loss.item():.4f}")


train_gru(gru, n_steps=300, batch_size=64)

gru_acc = evaluate(gru)
print(f"GRU Test Accuracy:          {gru_acc:.4f}")

[GRU] step 0 | loss = 3.0005
[GRU] step 50 | loss = 2.9924
[GRU] step 100 | loss = 2.9970


In [19]:
class SimpleMambaLayer(nn.Module):
    def __init__(self, d_model: int, state_dim: int):
        super().__init__()
        self.d_model = d_model
        self.state_dim = state_dim

        # Project input tokens into "state space"
        self.in_proj = nn.Linear(d_model, state_dim)

        # Input-dependent gating and mixing (selectivity)
        self.forget_gate = nn.Linear(d_model, state_dim)   # controls how much of old state to keep
        self.input_gate = nn.Linear(d_model, state_dim)    # controls how much new info to add

        # Project state back to model dimension
        self.state_norm = nn.LayerNorm(state_dim)
        self.out_proj = nn.Linear(state_dim, d_model)
        self.out_norm = nn.LayerNorm(d_model)

    def forward(self, x):
        """
        x: (B, T, d_model)
        returns: (B, T, d_model)
        """
        B, T, D = x.shape
        device = x.device

        # Initialize state (B, state_dim)
        s = torch.zeros(B, self.state_dim, device=device)

        outputs = []

        for t in range(T):
            x_t = x[:, t, :]  # (B, d_model)

            # Map input into state space
            u_t = self.in_proj(x_t)  # (B, state_dim)

            # Compute input-dependent gates
            f_t = torch.sigmoid(self.forget_gate(x_t))   # (B, state_dim) in (0,1)
            i_t = torch.sigmoid(self.input_gate(x_t))    # (B, state_dim)

            # State update: selective mixing of old state and new input
            s = f_t * s + i_t * u_t                      # (B, state_dim)
            s_norm = self.state_norm(s)

            # Project back to model dimension
            y_t = self.out_proj(s_norm)                       # (B, d_model)
            y_t = self.out_norm(y_t + x_t)
            outputs.append(y_t.unsqueeze(1))

        return torch.cat(outputs, dim=1)  # (B, T, d_model)


class SimpleMambaModel(nn.Module):
    def __init__(self, d_model: int = 64, state_dim: int = 128, num_layers: int = 2):
        super().__init__()
        self.embed = nn.Embedding(VOCAB_SIZE, d_model)

        layers = []
        for _ in range(num_layers):
            layers.append(SimpleMambaLayer(d_model, state_dim))
        self.layers = nn.ModuleList(layers)

        self.out = nn.Linear(d_model, VOCAB_SIZE)

    def forward(self, x):
        # x: (B, T)
        h = self.embed(x)  # (B, T, d_model)
        for layer in self.layers:
            h = layer(h)
        logits = self.out(h)  # (B, T, vocab)
        return logits

mamba = SimpleMambaModel(d_model=64, state_dim=128, num_layers=2).to(device)
opt_mamba = torch.optim.Adam(mamba.parameters(), lr=1e-3)

def train_mamba(model, n_steps: int = 300, batch_size: int = 64):
    model.train()
    for step in range(n_steps):
        idx = torch.randint(0, TRAIN_SAMPLES, (batch_size,))
        batch_x = X_train[idx].to(device)
        batch_y = Y_train[idx].to(device)

        logits = model(batch_x)
        loss = compute_loss_on_valid_positions(logits, batch_y)

        opt_mamba.zero_grad()
        loss.backward()
        opt_mamba.step()

        if step % 50 == 0:
            print(f"[Mamba-like] step {step} | loss = {loss.item():.4f}")


train_mamba(mamba, n_steps=300, batch_size=64)

mamba_acc = evaluate(mamba)
print(f"Simple Mamba-like Test Accuracy: {mamba_acc:.4f}")

[Mamba-like] step 0 | loss = 3.1541
[Mamba-like] step 50 | loss = 2.9977
[Mamba-like] step 100 | loss = 2.9990
[Mamba-like] step 150 | loss = 2.9974
[Mamba-like] step 200 | loss = 2.9965
[Mamba-like] step 250 | loss = 2.9961
Simple Mamba-like Test Accuracy: 0.0487
